# Digital twin with tools and evaluation — build a grounded business assistant, hand-build its agent loop, and review each reply with a second LLM call.


## 1. Setup — DRY_RUN uses an offline client and prevents file writes; set it to False to use OpenAI and save visitor emails.


In [ ]:
import json
import re
from pathlib import Path
from types import SimpleNamespace as NS

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path("../../../.env"), override=True)
DRY_RUN = True
GEN_MODEL = "gpt-5.4-mini"
EVAL_MODEL = "gpt-5.4-nano"


## 2. Ground the assistant — inline fictional business facts keep the lab self-contained and free of personal data.


In [ ]:
BUSINESS_FACTS = """Alex Example runs Example Data Studio, a fictional data and AI consultancy.
Services: data pipeline audits, dashboard design, and AI workflow prototypes.
A typical prototype engagement lasts four weeks and is delivered remotely.
Pricing, client names, and current availability are not provided; say you do not know.
Visitors who want follow-up can leave their own email address."""

SYSTEM_PROMPT = f"""You are the website assistant for Example Data Studio.
Answer only work-related questions using the business facts below.
If an answer is missing, say you do not know; never invent details.
For unrelated requests, politely redirect to the business.
When a visitor provides their own email for follow-up, call record_email.
Do not repeat email addresses in replies.

# Business facts
{BUSINESS_FACTS}"""


## 3. Define a tool — the function, its JSON schema, and a dispatch table are the three pieces the agent loop needs.


In [ ]:
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

def record_email(email):
    email = email.strip().lower()
    if not EMAIL_RE.fullmatch(email):
        return "Invalid email; ask the visitor to check it."
    if DRY_RUN:
        return "DRY RUN: valid email received; nothing was written. Explain this to the visitor."
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email recorded; confirm this without repeating the address."

record_email_json = {
    "name": "record_email",
    "description": "Record a visitor's own email address when they request follow-up.",
    "parameters": {
        "type": "object",
        "properties": {"email": {"type": "string", "description": "The visitor's email address"}},
        "required": ["email"],
        "additionalProperties": False,
    },
}
tools = [{"type": "function", "function": record_email_json}]
TOOL_DISPATCH = {"record_email": record_email}


## 4. Hand-build the agent loop — append every tool call and result, then call the model again until it produces a reply.


In [ ]:
def run_tool(tool_call):
    arguments = json.loads(tool_call.function.arguments)
    return TOOL_DISPATCH[tool_call.function.name](**arguments)

def generate(message, history, feedback=None):
    history = [{"role": m["role"], "content": m["content"]} for m in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if feedback:
        messages.append({"role": "system", "content": f"Previous draft failed work-only review: {feedback}. Fix this without mentioning the review."})
    messages += history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=GEN_MODEL, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        model_message = response.choices[0].message
        messages.append(model_message)
        for tool_call in model_message.tool_calls:
            result = run_tool(tool_call)
            messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model=GEN_MODEL, messages=messages, tools=tools)
    return response.choices[0].message.content


## 5. Evaluate and optimise — a separate LLM call returns JSON; an off-topic draft gets one feedback-guided regeneration.


In [ ]:
EVALUATOR_PROMPT = """You review replies for Example Data Studio.
Return JSON with work_related (boolean) and reason (string).
Pass only replies about the business, its data and AI services, or polite redirects to those topics.
Fail replies that answer unrelated questions, even when the answer is correct."""

def evaluate(message, draft):
    review = f"Visitor message: {message}\nDraft reply: {draft}"
    response = openai.chat.completions.create(
        model=EVAL_MODEL,
        messages=[{"role": "system", "content": EVALUATOR_PROMPT}, {"role": "user", "content": review}],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

def chat(message, history):
    feedback = None
    for _ in range(2):
        draft = generate(message, history, feedback)
        verdict = evaluate(message, draft)
        if verdict["work_related"]:
            return draft
        feedback = verdict["reason"]
    return "I can help only with Example Data Studio's services and work-related questions."


## 6. Exercise the whole loop offline — this stub returns a tool call, a business reply, an off-topic draft, and evaluator verdicts.


In [ ]:
def response(content, reason="stop", calls=None):
    message = NS(content=content, tool_calls=calls)
    return NS(choices=[NS(message=message, finish_reason=reason)])

class FakeCompletions:
    def create(self, model, messages, **kwargs):
        if kwargs.get("response_format"):
            ok = "Paris" not in messages[-1]["content"]
            verdict = {"work_related": ok, "reason": "" if ok else "The draft answers an unrelated question."}
            return response(json.dumps(verdict))
        last = messages[-1]
        if last["role"] == "user" and "visitor@example.com" in last["content"]:
            function = NS(name="record_email", arguments='{"email": "visitor@example.com"}')
            return response(None, "tool_calls", [NS(id="call-1", function=function)])
        if any(isinstance(m, dict) and m["role"] == "tool" for m in messages):
            return response("DRY RUN: your email was validated, but nothing was saved.")
        user = next(m["content"] for m in reversed(messages) if isinstance(m, dict) and m["role"] == "user")
        reviewed = any(isinstance(m, dict) and "Previous draft failed" in m.get("content", "") for m in messages)
        if "capital" in user.lower() and not reviewed:
            return response("Paris is the capital of France.")
        if "capital" in user.lower():
            return response("I can help only with Example Data Studio's services and work-related questions.")
        return response("Example Data Studio designs data pipelines, dashboards, and AI workflow prototypes.")

openai = NS(chat=NS(completions=FakeCompletions())) if DRY_RUN else OpenAI()


In [ ]:
if DRY_RUN:
    business_reply = chat("What services does the studio offer?", [])
    email_reply = chat("Please follow up at visitor@example.com", [])
    off_topic_reply = chat("What is the capital of France?", [])
    assert "data" in business_reply.lower()
    assert "dry run" in email_reply.lower()
    assert "Paris" not in off_topic_reply
    print("Offline checks passed: grounded reply, email tool, and off-topic regeneration.")


## 7. Chat UI — the same chat function supplies conversation history; the server launches only in live mode.


In [ ]:
if not DRY_RUN:
    import gradio as gr
    demo = gr.ChatInterface(chat, title="Example Data Studio", description="Ask work-related questions about the fictional business.")
    demo.launch(inbrowser=True)
